In [18]:
# Load and check the features
import duckdb
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

data = duckdb.sql(
    "SELECT * FROM read_parquet('../data/gold/nyc/model_table.parquet')"
).df()

# Share of empty values in each column
print((data.isna().mean() * 100).round(1).sort_values(ascending=False).head(12))

instant_bookable     100.0
host_years           100.0
bedrooms              29.5
rating_overall        28.2
beds                   8.1
host_is_superhost      1.3
bathrooms              0.1
price_outlier          0.0
listing_id             0.0
host_id                0.0
base_price             0.0
log_base_price         0.0
dtype: float64


In [19]:
# Split by host
# Train and test only on non-outlier listings
usable = data[data["price_outlier"] == False].reset_index(drop=True)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(usable, groups=usable["host_id"]))

train = usable.iloc[train_idx].reset_index(drop=True)
test = usable.iloc[test_idx].reset_index(drop=True)

print("Train listings:", len(train))
print("Test listings: ", len(test))
print("Hosts in both: ", len(set(train["host_id"]) & set(test["host_id"])))

# Save the split so every model uses exactly the same test listings
split = pd.concat([
    train[["listing_id"]].assign(split="train"),
    test[["listing_id"]].assign(split="test"),
])
split.to_parquet("../data/gold/nyc/split.parquet", index=False)

Train listings: 17035
Test listings:  4378
Hosts in both:  0


In [20]:
# How we measure errors
results = []

def evaluate(name, actual, predicted):
    """Compare predicted and actual base prices (both in dollars)."""
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    missing = np.isnan(predicted)
    if missing.any():
        print(f"[{name}] WARNING: {missing.sum()} listings had no prediction "
              f"and were excluded from the error metrics")

    actual, predicted = actual[~missing], predicted[~missing]
    pct_error = np.abs(predicted - actual) / actual * 100

    row = {
        "model": name,
        "n_evaluated": len(actual),
        "median_abs_pct_error": round(np.median(pct_error), 1),
        "within_20pct": round((pct_error <= 20).mean() * 100, 1),
        "median_abs_error_$": round(np.median(np.abs(predicted - actual)), 1),
    }
    results.append(row)
    return row

In [21]:
# Which room type + stay type combos exist in test but not in train?
train_combos = set(map(tuple, train[["room_type", "stay_type"]].drop_duplicates().values))
test_combos = set(map(tuple, test[["room_type", "stay_type"]].drop_duplicates().values))

missing = test_combos - train_combos
print("Combos in test but missing from train:", missing)

for room, stay in missing:
    count = ((test["room_type"] == room) & (test["stay_type"] == stay)).sum()
    print(f"  {room} / {stay}: {count} test listings")

Combos in test but missing from train: {('Hotel room', 'monthly stay')}
  Hotel room / monthly stay: 21 test listings


In [22]:
#Baseline 1, the segment median
# The simplest rule: predict the median base price of the listing's
# room type + stay type, calculated from TRAINING listings only
segment_median = (
    train.groupby(["room_type", "stay_type"])["base_price"]
    .median()
    .rename("prediction")
    .reset_index()
)

pred = test.merge(segment_median, on=["room_type", "stay_type"], how="left")["prediction"]
evaluate("baseline: segment median", test["base_price"], pred)

[baseline: segment median] WARNING: 21 listings had no prediction and were excluded from the error metrics


{'model': 'baseline: segment median',
 'n_evaluated': 4357,
 'median_abs_pct_error': np.float64(39.7),
 'within_20pct': np.float64(26.6),
 'median_abs_error_$': np.float64(76.3)}

In [23]:
# Baseline 2, the comparable-listing median
comps = duckdb.sql(
    "SELECT listing_id, comp_median FROM read_parquet('../data/gold/nyc/comps.parquet')"
).df()

pred = test.merge(comps, on="listing_id", how="left")["comp_median"]
evaluate("baseline: comp median (Milestone 4)", test["base_price"], pred)

pd.DataFrame(results)

,model,n_evaluated,median_abs_pct_error,within_20pct,median_abs_error_$
0,baseline: segment median,4357,39.7,26.6,76.3
1,baseline: comp median (Milestone 4),4378,30.1,35.3,57.0
